In [ ]:
import os

os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

## 1) Imports

In [ ]:
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import copy
import torch.nn.functional as F

from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from safetensors.torch import load_file as safe_load_file
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

from peft import LoraConfig, get_peft_model, TaskType

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
RUN_NAME = "cifar100_5step_e15_10_5_15_n3"

RESULTS_DIR = os.path.join("results", RUN_NAME)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

STEP1_OUT = f"outputs/{RUN_NAME}/step1_scratch"
STEP1_FINAL_OUT = f"outputs/{RUN_NAME}/step1_scratch_final"
JOINT_OUT = f"outputs/{RUN_NAME}/joint_full"
LORA_BANK_DIR = f"outputs/{RUN_NAME}/lora_bank"
ZIP_MERGE_DIR = f"outputs/{RUN_NAME}/zip_merge"
ZIP_FINAL_DIR = os.path.join(ZIP_MERGE_DIR, "final_model")

os.makedirs(LORA_BANK_DIR, exist_ok=True)
os.makedirs(ZIP_MERGE_DIR, exist_ok=True)
os.makedirs(ZIP_FINAL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(f"outputs/{RUN_NAME}", exist_ok=True)

## 2) Load CIFAR-100 (fine labels)

In [ ]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

In [ ]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_dataset_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [ ]:

num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

In [ ]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))

## 4) Model + preprocessing

In [ ]:
# Requested model
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

## 5) Build per-step datasets (step / cumulative / full)

In [ ]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

## 6) Training recipes (reasonable settings)

In [ ]:
set_seed(42)

SCRATCH_EPOCHS = 1
LORA_EPOCHS    = 1
FT_EPOCHS      = 1
JOINT_EPOCHS   = 1

# SCRATCH_EPOCHS = 15
# LORA_EPOCHS    = 10
# FT_EPOCHS      = 5
# JOINT_EPOCHS   = 15

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

LR_SCRATCH = 5e-5
LR_LORA    = 5e-5
LR_FT      = 3e-5
LR_JOINT   = 5e-5

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

results = []

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "init_mode": "scratch",
    "num_classes": num_classes,
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,
    "scratch_epochs": SCRATCH_EPOCHS,
    "lora_epochs": LORA_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "joint_epochs": JOINT_EPOCHS,
    "lr_scratch": LR_SCRATCH,
    "lr_lora": LR_LORA,
    "lr_ft": LR_FT,
    "lr_joint": LR_JOINT,
}
with open(os.path.join(METRICS_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

## 7) Step 1: train full ViT from scratch on step 0 classes

In [ ]:
import os, shutil, json

# --- clean old step1 outputs so stale checkpoints cannot survive ---
if os.path.exists(STEP1_OUT):
    shutil.rmtree(STEP1_OUT)
if os.path.exists(STEP1_FINAL_OUT):
    shutil.rmtree(STEP1_FINAL_OUT)

os.makedirs(STEP1_OUT, exist_ok=True)
os.makedirs(STEP1_FINAL_OUT, exist_ok=True)

step1_idx = 0
train_step1, test_step1, label2new_1, new2orig_1, class_ids_1 = make_step_datasets(
    step1_idx, split_type="new_only", remap_labels=False
)

print("Step1 original classes:", class_ids_1[:5], "...", class_ids_1[-5:])
print(
    "Step1 label range:",
    min(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
    max(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
)

config_step1 = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

model_step1 = AutoModelForImageClassification.from_config(config_step1)

print("Before training - Step1 model num_labels:", model_step1.config.num_labels)
print("Before training - Step1 classifier out_features:", model_step1.classifier.out_features)
assert model_step1.config.num_labels == num_classes
assert model_step1.classifier.out_features == num_classes

args_step1 = TrainingArguments(
    output_dir=STEP1_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=SCRATCH_EPOCHS,
    learning_rate=LR_SCRATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_SCRATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_SCRATCH,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    max_grad_norm=1.0,
)

trainer_step1 = Trainer(
    model=model_step1,
    args=args_step1,
    train_dataset=train_step1,
    eval_dataset=test_step1,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_step1.remove_callback(NotebookProgressCallback)

trainer_step1.train()

eval_step1 = trainer_step1.evaluate()
print({"eval_step1": eval_step1})

print("About to save B0 final model to:", STEP1_FINAL_OUT)
print("Trainer model type:", type(trainer_step1.model))
print("Trainer model class:", trainer_step1.model.__class__.__name__)

trainer_step1.model.save_pretrained(STEP1_FINAL_OUT, safe_serialization=False)
image_processor.save_pretrained(STEP1_FINAL_OUT)

print("Saved STEP1_FINAL_OUT to:", STEP1_FINAL_OUT)
print("STEP1_FINAL_OUT exists:", os.path.exists(STEP1_FINAL_OUT))
print("Files in STEP1_FINAL_OUT:", os.listdir(STEP1_FINAL_OUT) if os.path.exists(STEP1_FINAL_OUT) else "MISSING")

cfg_path = os.path.join(STEP1_FINAL_OUT, "config.json")
print("config exists:", os.path.exists(cfg_path))

if os.path.exists(cfg_path):
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    print("model_type:", cfg.get("model_type"))
    print("architectures:", cfg.get("architectures"))
    print("num_labels:", cfg.get("num_labels"))

reloaded_step1 = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
print("Reloaded STEP1 num_labels:", reloaded_step1.config.num_labels)
print("Reloaded STEP1 classifier out_features:", reloaded_step1.classifier.out_features)

assert reloaded_step1.config.num_labels == num_classes
assert reloaded_step1.classifier.out_features == num_classes

In [ ]:
print("Init mode:", run_config["init_mode"])
assert run_config["init_mode"] == "scratch"

In [ ]:
step1_log_df = pd.DataFrame(trainer_step1.state.log_history)
step1_log_df.to_csv(os.path.join(TABLES_DIR, "step1_log_history.csv"), index=False)

step1_metrics = {
    "experiment": "step1_scratch",
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "step1_metrics.json"), "w") as f:
    json.dump(step1_metrics, f, indent=2)

results.append({
    "experiment": "step1_scratch",
    "method": "full_finetune",
    "step": 1,
    "eval_type": "current_step",
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
})

step1_log_df.tail()

In [ ]:
if "loss" in step1_log_df.columns:
    train_loss_df = step1_log_df[step1_log_df["loss"].notna()]
    if not train_loss_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(train_loss_df["step"], train_loss_df["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train Loss")
        plt.title("Step 1 Train Loss")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_train_loss.png"), dpi=200)
        plt.show()

if "eval_accuracy" in step1_log_df.columns:
    eval_df = step1_log_df[step1_log_df["eval_accuracy"].notna()]
    if not eval_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Eval Accuracy")
        plt.title("Step 1 Eval Accuracy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_eval_accuracy.png"), dpi=200)
        plt.show()

## 8) Step 2: LoRA only (freeze backbone) on top of Step 1 model

In [ ]:
first_step_classes = classes_for_step(0)

def make_eval_dataset(class_ids, name=None):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    if name is not None:
        print(f"[Eval set] {name}: num_classes={len(class_ids)}, size={len(test_ds)}")
    return test_ds

In [ ]:
lora_results = []

for s in range(2, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_lora"
    stale_adapter_dir = os.path.join(LORA_BANK_DIR, f"step_{s}_adapter")

    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_adapter_dir):
        shutil.rmtree(stale_adapter_dir)

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

# every LoRA adapter is trained on the SAME fixed backbone
base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print(f"\n[LoRA separate] Step {step_idx+1}")
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    tr_min, tr_max = _label_range(train_step)
    te_min, te_max = _label_range(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)
    print("Num labels:", num_classes)

    print("Loaded fixed base_model_dir:", base_model_dir)
    base_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print("Loaded base_model num_labels:", base_model.config.num_labels)
    print("Loaded base_model classifier out_features:", base_model.classifier.out_features)

    assert base_model.config.num_labels == num_classes, (
        f"Loaded base model num_labels={base_model.config.num_labels}, expected {num_classes}"
    )
    assert base_model.classifier.out_features == num_classes, (
        f"Loaded base classifier out_features={base_model.classifier.out_features}, expected {num_classes}"
    )

    assert tr_min >= 0
    assert tr_max < base_model.classifier.out_features, (
        f"Train labels out of range: max label {tr_max}, classifier out_features {base_model.classifier.out_features}"
    )
    assert te_min >= 0
    assert te_max < base_model.classifier.out_features, (
        f"Test labels out of range: max label {te_max}, classifier out_features {base_model.classifier.out_features}"
    )

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        bias="none",
        target_modules=["query", "key", "value"],
        modules_to_save=["classifier"],
    )

    model_lora = get_peft_model(base_model, lora_config)

    print(f"[LoRA separate] Wrapped model step {step_idx+1} num_labels:", model_lora.config.num_labels)
    print(f"[LoRA separate] Wrapped model step {step_idx+1} classifier out_features:", model_lora.classifier.out_features)

    assert model_lora.config.num_labels == num_classes, (
        f"Wrapped LoRA model num_labels={model_lora.config.num_labels}, expected {num_classes}"
    )
    assert model_lora.classifier.out_features == num_classes, (
        f"Wrapped LoRA classifier out_features={model_lora.classifier.out_features}, expected {num_classes}"
    )

    model_lora.print_trainable_parameters()

    step_lora_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_lora"

    args_lora = TrainingArguments(
        output_dir=step_lora_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=LORA_EPOCHS,
        learning_rate=LR_LORA,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_LORA,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_LORA,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_lora = Trainer(
        model=model_lora,
        args=args_lora,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_lora.train()
    eval_current = trainer_lora.evaluate(test_step)

    eval_first = make_eval_dataset(classes_for_step(0))
    seen_now = classes_for_cumulative(step_idx)
    later_now = [c for c in seen_now if c not in classes_for_step(0)]
    
    eval_later = make_eval_dataset(later_now) if len(later_now) > 0 else None
    eval_seen  = make_eval_dataset(seen_now)
    
    eval_first_res = trainer_lora.evaluate(eval_dataset=eval_first)
    eval_later_res = trainer_lora.evaluate(eval_dataset=eval_later) if eval_later is not None else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    eval_seen_res  = trainer_lora.evaluate(eval_dataset=eval_seen)

    pd.DataFrame(trainer_lora.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_lora_log_history.csv"),
        index=False
    )

    # save adapter only
    step_adapter_dir = os.path.join(LORA_BANK_DIR, f"step_{step_idx+1}_adapter")
    os.makedirs(step_adapter_dir, exist_ok=True)

    print(f"[LoRA separate] Step {step_idx+1} saving adapter to {step_adapter_dir}")
    trainer_lora.model.save_pretrained(step_adapter_dir)
    image_processor.save_pretrained(step_adapter_dir)

    adapter_meta = {
        "step": step_idx + 1,
        "base_model_dir": STEP1_FINAL_OUT,
        "class_ids": class_ids,
        "num_labels": num_classes,
        "strategy": "separate_adapter_no_backbone_merge"
    }
    with open(os.path.join(step_adapter_dir, "adapter_meta.json"), "w") as f:
        json.dump(adapter_meta, f, indent=2)

    print(f"[LoRA separate] Step {step_idx+1} adapter save finished")

    lora_results.extend([
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_separate",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_separate",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_first_res.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_separate",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_later_res.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_separate",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_seen_res.get("eval_loss", np.nan)),
        },
    ])

print("\nLoRA separate-adapter training finished.")

In [ ]:
# Base model B0
B0_dir = STEP1_FINAL_OUT

# Saved adapter dirs
L1_dir = os.path.join(LORA_BANK_DIR, "step_2_adapter")
L2_dir = os.path.join(LORA_BANK_DIR, "step_3_adapter")
L3_dir = os.path.join(LORA_BANK_DIR, "step_4_adapter")
L4_dir = os.path.join(LORA_BANK_DIR, "step_5_adapter")

print("B0_dir:", B0_dir)
print("L1_dir:", L1_dir)
print("L2_dir:", L2_dir)
print("L3_dir:", L3_dir)
print("L4_dir:", L4_dir)

def extract_lora_tensors(adapter_dir):
    base_model = AutoModelForImageClassification.from_pretrained(B0_dir)
    peft_model = PeftModel.from_pretrained(base_model, adapter_dir)

    lora_state = {}
    for k, v in peft_model.state_dict().items():
        if "lora_" in k:
            lora_state[k] = v.detach().cpu().clone()

    return lora_state


L1 = extract_lora_tensors(L1_dir)
L2 = extract_lora_tensors(L2_dir)
L3 = extract_lora_tensors(L3_dir)
L4 = extract_lora_tensors(L4_dir)

print("num tensors in L1:", len(L1))
print("num tensors in L2:", len(L2))
print("num tensors in L3:", len(L3))
print("num tensors in L4:", len(L4))

In [ ]:
def filter_dataset_by_classes(dataset, class_ids):
    class_ids = set(class_ids)
    return dataset.filter(lambda x: x["label"] in class_ids)

In [ ]:
assert set(L1.keys()) == set(L2.keys()) == set(L3.keys()) == set(L4.keys())
print("All adapters have the same LoRA keys.")

## merging method

In [ ]:
import os
import torch
from peft import PeftConfig

print("\n===== FIXED AVG MERGE: delta-space backbone + stitched classifier =====")

ADAPTER_DIRS = [L1_dir, L2_dir, L3_dir, L4_dir]
ADAPTER_CLASS_GROUPS = [
    classes_for_step(1),  # 20..39
    classes_for_step(2),  # 40..59
    classes_for_step(3),  # 60..79
    classes_for_step(4),  # 80..99
]

def load_adapter_state(adapter_dir):
    safe_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    bin_path = os.path.join(adapter_dir, "adapter_model.bin")
    if os.path.exists(safe_path):
        from safetensors.torch import load_file
        return load_file(safe_path)
    elif os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter weights found in {adapter_dir}")

def get_lora_scaling_factor(adapter_dir):
    peft_cfg = PeftConfig.from_pretrained(adapter_dir)
    return peft_cfg.lora_alpha / peft_cfg.r

def normalize_module_name(module_name):
    prefixes = [
        "base_model.model.",
        "base_model.",
        "model.",
    ]
    for p in prefixes:
        if module_name.startswith(p):
            module_name = module_name[len(p):]
    module_name = module_name.replace("vit.model.", "vit.")
    return module_name

def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)
    submodule = model
    for part in module_name.split("."):
        submodule = getattr(submodule, part)
    return submodule

def extract_lora_factor_matrices(adapter_state):
    lora_factor_matrices = {}

    for param_name, param_tensor in adapter_state.items():
        if ".lora_A." in param_name:
            base_module_name = param_name.split(".lora_A.")[0]
            lora_factor_matrices.setdefault(base_module_name, {})["A_matrix"] = param_tensor.detach().cpu().float()

        elif ".lora_B." in param_name:
            base_module_name = param_name.split(".lora_B.")[0]
            lora_factor_matrices.setdefault(base_module_name, {})["B_matrix"] = param_tensor.detach().cpu().float()

    lora_factor_matrices = {
        module_name: factors
        for module_name, factors in lora_factor_matrices.items()
        if "A_matrix" in factors and "B_matrix" in factors
    }

    return lora_factor_matrices

def extract_classifier_tensors(adapter_state):
    classifier_weight_key = None
    classifier_bias_key = None

    for param_name in adapter_state.keys():
        if "classifier" in param_name and param_name.endswith("weight"):
            classifier_weight_key = param_name
        if "classifier" in param_name and param_name.endswith("bias"):
            classifier_bias_key = param_name

    if classifier_weight_key is None:
        return None, None

    classifier_weight_tensor = adapter_state[classifier_weight_key].detach().cpu().float()
    classifier_bias_tensor = (
        adapter_state[classifier_bias_key].detach().cpu().float()
        if classifier_bias_key is not None else None
    )

    return classifier_weight_tensor, classifier_bias_tensor

def build_fixed_avg_merged_model(base_model_dir, adapter_dirs, adapter_class_groups):
    base_model_W0 = AutoModelForImageClassification.from_pretrained(base_model_dir)
    base_model_W0.eval()

    module_to_delta_list = {}
    classifier_row_sources = []

    for adapter_dir, class_ids_for_this_adapter in zip(adapter_dirs, adapter_class_groups):
        adapter_state = load_adapter_state(adapter_dir)
        lora_scaling_factor_si = get_lora_scaling_factor(adapter_dir)
        module_to_factor_matrices = extract_lora_factor_matrices(adapter_state)

        for module_name, factor_dict in module_to_factor_matrices.items():
            A_i = factor_dict["A_matrix"]
            B_i = factor_dict["B_matrix"]

            effective_lora_delta_Delta_i = (B_i @ A_i) * lora_scaling_factor_si
            module_to_delta_list.setdefault(module_name, []).append(effective_lora_delta_Delta_i)

        classifier_weight_i, classifier_bias_i = extract_classifier_tensors(adapter_state)
        classifier_row_sources.append(
            (class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i)
        )

    with torch.no_grad():
        for module_name, delta_list_for_this_module in module_to_delta_list.items():
            average_delta_Delta_avg = torch.stack(delta_list_for_this_module, dim=0).mean(dim=0)
            target_weight_module = get_submodule_by_name(base_model_W0, module_name)

            if target_weight_module.weight.shape != average_delta_Delta_avg.shape:
                raise ValueError(
                    f"Shape mismatch for {module_name}: "
                    f"weight={target_weight_module.weight.shape}, "
                    f"delta={average_delta_Delta_avg.shape}"
                )

            target_weight_module.weight.data += average_delta_Delta_avg.to(
                target_weight_module.weight.device,
                dtype=target_weight_module.weight.dtype
            )

    with torch.no_grad():
        for class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i in classifier_row_sources:
            if classifier_weight_i is None:
                continue

            base_model_W0.classifier.weight.data[class_ids_for_this_adapter] = classifier_weight_i[class_ids_for_this_adapter].to(
                base_model_W0.classifier.weight.device,
                dtype=base_model_W0.classifier.weight.dtype
            )

            if classifier_bias_i is not None and base_model_W0.classifier.bias is not None:
                base_model_W0.classifier.bias.data[class_ids_for_this_adapter] = classifier_bias_i[class_ids_for_this_adapter].to(
                    base_model_W0.classifier.bias.device,
                    dtype=base_model_W0.classifier.bias.dtype
                )

    base_model_W0.eval()
    return base_model_W0

model_L1234_avg = build_fixed_avg_merged_model(
    base_model_dir=B0_dir,
    adapter_dirs=ADAPTER_DIRS,
    adapter_class_groups=ADAPTER_CLASS_GROUPS,
)

print("Built FIXED AVG model_L1234_avg")
print("Classifier out_features:", model_L1234_avg.classifier.out_features)

first_step_classes = classes_for_step(0)
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))
final_seen_classes = classes_for_cumulative(num_steps - 1)

fixed_avg_eval_first = make_eval_dataset(first_step_classes, name="fixed_avg_first")
fixed_avg_eval_later = make_eval_dataset(later_step_classes, name="fixed_avg_later")
fixed_avg_eval_seen  = make_eval_dataset(final_seen_classes, name="fixed_avg_seen")

trainer_eval_fixed_avg = Trainer(
    model=model_L1234_avg,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_fixed_avg_final",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

fixed_avg_final_first = trainer_eval_fixed_avg.evaluate(eval_dataset=fixed_avg_eval_first)
fixed_avg_final_later = trainer_eval_fixed_avg.evaluate(eval_dataset=fixed_avg_eval_later)
fixed_avg_final_seen  = trainer_eval_fixed_avg.evaluate(eval_dataset=fixed_avg_eval_seen)

print("Fixed AVG Merged LoRA final - first_step:", fixed_avg_final_first)
print("Fixed AVG Merged LoRA final - later_steps:", fixed_avg_final_later)
print("Fixed AVG Merged LoRA final - all_seen:", fixed_avg_final_seen)

In [ ]:
import os
import torch
from peft import PeftConfig

print("\n===== FIXED AVG + TIES MERGE: delta-space backbone + stitched classifier =====")

ADAPTER_DIRS = [L1_dir, L2_dir, L3_dir, L4_dir]
ADAPTER_CLASS_GROUPS = [
    classes_for_step(1),  # 20..39
    classes_for_step(2),  # 40..59
    classes_for_step(3),  # 60..79
    classes_for_step(4),  # 80..99
]

# -----------------------------
# TIES hyperparameters
# -----------------------------
# density = fraction of delta entries kept after TRIM
TIES_DENSITY = 0.20
# sign election method: "sum" or "majority"
TIES_SIGN_MODE = "sum"

def load_adapter_state(adapter_dir):
    safe_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    bin_path = os.path.join(adapter_dir, "adapter_model.bin")
    if os.path.exists(safe_path):
        from safetensors.torch import load_file
        return load_file(safe_path)
    elif os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter weights found in {adapter_dir}")

def get_lora_scaling_factor(adapter_dir):
    peft_cfg = PeftConfig.from_pretrained(adapter_dir)
    return peft_cfg.lora_alpha / peft_cfg.r

def normalize_module_name(module_name):
    prefixes = [
        "base_model.model.",
        "base_model.",
        "model.",
    ]
    for p in prefixes:
        if module_name.startswith(p):
            module_name = module_name[len(p):]
    module_name = module_name.replace("vit.model.", "vit.")
    return module_name

def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)
    submodule = model
    for part in module_name.split("."):
        submodule = getattr(submodule, part)
    return submodule

def extract_lora_factor_matrices(adapter_state):
    lora_factor_matrices = {}

    for param_name, param_tensor in adapter_state.items():
        if ".lora_A." in param_name:
            base_module_name = param_name.split(".lora_A.")[0]
            lora_factor_matrices.setdefault(base_module_name, {})["A_matrix"] = param_tensor.detach().cpu().float()

        elif ".lora_B." in param_name:
            base_module_name = param_name.split(".lora_B.")[0]
            lora_factor_matrices.setdefault(base_module_name, {})["B_matrix"] = param_tensor.detach().cpu().float()

    lora_factor_matrices = {
        module_name: factors
        for module_name, factors in lora_factor_matrices.items()
        if "A_matrix" in factors and "B_matrix" in factors
    }

    return lora_factor_matrices

def extract_classifier_tensors(adapter_state):
    classifier_weight_key = None
    classifier_bias_key = None

    for param_name in adapter_state.keys():
        if "classifier" in param_name and param_name.endswith("weight"):
            classifier_weight_key = param_name
        if "classifier" in param_name and param_name.endswith("bias"):
            classifier_bias_key = param_name

    if classifier_weight_key is None:
        return None, None

    classifier_weight_tensor = adapter_state[classifier_weight_key].detach().cpu().float()
    classifier_bias_tensor = (
        adapter_state[classifier_bias_key].detach().cpu().float()
        if classifier_bias_key is not None else None
    )

    return classifier_weight_tensor, classifier_bias_tensor


# ============================================================
# Fixed AVG merge
# ============================================================
def build_fixed_avg_merged_model(base_model_dir, adapter_dirs, adapter_class_groups):
    base_model_W0 = AutoModelForImageClassification.from_pretrained(base_model_dir)
    base_model_W0.eval()

    module_to_delta_list = {}
    classifier_row_sources = []

    for adapter_dir, class_ids_for_this_adapter in zip(adapter_dirs, adapter_class_groups):
        adapter_state = load_adapter_state(adapter_dir)
        lora_scaling_factor_si = get_lora_scaling_factor(adapter_dir)
        module_to_factor_matrices = extract_lora_factor_matrices(adapter_state)

        for module_name, factor_dict in module_to_factor_matrices.items():
            A_i = factor_dict["A_matrix"]
            B_i = factor_dict["B_matrix"]
            delta_i = (B_i @ A_i) * lora_scaling_factor_si
            module_to_delta_list.setdefault(module_name, []).append(delta_i)

        classifier_weight_i, classifier_bias_i = extract_classifier_tensors(adapter_state)
        classifier_row_sources.append((class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i))

    with torch.no_grad():
        for module_name, delta_list_for_this_module in module_to_delta_list.items():
            avg_delta = torch.stack(delta_list_for_this_module, dim=0).mean(dim=0)
            target_weight_module = get_submodule_by_name(base_model_W0, module_name)

            if target_weight_module.weight.shape != avg_delta.shape:
                raise ValueError(
                    f"Shape mismatch for {module_name}: "
                    f"weight={target_weight_module.weight.shape}, delta={avg_delta.shape}"
                )

            target_weight_module.weight.data += avg_delta.to(
                target_weight_module.weight.device,
                dtype=target_weight_module.weight.dtype
            )

        # stitched classifier
        if hasattr(base_model_W0, "classifier"):
            for class_ids_for_this_adapter, clf_w_i, clf_b_i in classifier_row_sources:
                if clf_w_i is None:
                    continue

                for row_idx in class_ids_for_this_adapter:
                    base_model_W0.classifier.weight.data[row_idx] = clf_w_i[row_idx].to(
                        base_model_W0.classifier.weight.device,
                        dtype=base_model_W0.classifier.weight.dtype
                    )
                    if clf_b_i is not None and base_model_W0.classifier.bias is not None:
                        base_model_W0.classifier.bias.data[row_idx] = clf_b_i[row_idx].to(
                            base_model_W0.classifier.bias.device,
                            dtype=base_model_W0.classifier.bias.dtype
                        )

    return base_model_W0


# ============================================================
# TIES merge helpers
# ============================================================
def ties_trim_topk(delta_tensor, density=0.20):
    """
    TRIM:
    keep only the top-|delta| entries according to density.
    """
    flat = delta_tensor.flatten()
    numel = flat.numel()
    k = max(1, int(density * numel))

    if k >= numel:
        return delta_tensor.clone()

    abs_flat = flat.abs()
    topk_vals, _ = torch.topk(abs_flat, k=k, largest=True, sorted=False)
    threshold = topk_vals.min()

    trimmed = delta_tensor.clone()
    trimmed[trimmed.abs() < threshold] = 0.0
    return trimmed

def ties_elect_sign(delta_stack, mode="sum"):
    """
    ELECT SIGN:
    delta_stack shape = [num_adapters, ...]
    Returns tensor in {-1, 0, +1}.
    """
    if mode == "sum":
        sign_source = delta_stack.sum(dim=0)
        elected = torch.sign(sign_source)
    elif mode == "majority":
        signs = torch.sign(delta_stack)
        pos_votes = (signs > 0).sum(dim=0)
        neg_votes = (signs < 0).sum(dim=0)
        elected = torch.zeros_like(delta_stack[0])
        elected[pos_votes > neg_votes] = 1.0
        elected[neg_votes > pos_votes] = -1.0
    else:
        raise ValueError(f"Unknown sign mode: {mode}")

    return elected

def ties_disjoint_merge(delta_list, density=0.20, sign_mode="sum"):
    """
    Paper-faithful TIES adaptation to LoRA deltas:
    1) TRIM each delta
    2) ELECT SIGN across trimmed deltas
    3) MERGE only deltas aligned with elected sign
    """
    trimmed_list = [ties_trim_topk(d, density=density) for d in delta_list]
    delta_stack = torch.stack(trimmed_list, dim=0)  # [n_adapters, ...]

    elected_sign = ties_elect_sign(delta_stack, mode=sign_mode)  # {-1,0,+1}

    aligned_mask = torch.sign(delta_stack) == elected_sign.unsqueeze(0)
    aligned_mask = aligned_mask & (elected_sign.unsqueeze(0) != 0)

    aligned_values = torch.where(aligned_mask, delta_stack, torch.zeros_like(delta_stack))
    aligned_count = aligned_mask.sum(dim=0).clamp(min=1)

    merged_delta = aligned_values.sum(dim=0) / aligned_count
    merged_delta[elected_sign == 0] = 0.0

    return merged_delta

def build_ties_merged_model(base_model_dir, adapter_dirs, adapter_class_groups,
                            density=0.20, sign_mode="sum"):
    base_model_W0 = AutoModelForImageClassification.from_pretrained(base_model_dir)
    base_model_W0.eval()

    module_to_delta_list = {}
    classifier_row_sources = []

    for adapter_dir, class_ids_for_this_adapter in zip(adapter_dirs, adapter_class_groups):
        adapter_state = load_adapter_state(adapter_dir)
        lora_scaling_factor_si = get_lora_scaling_factor(adapter_dir)
        module_to_factor_matrices = extract_lora_factor_matrices(adapter_state)

        for module_name, factor_dict in module_to_factor_matrices.items():
            A_i = factor_dict["A_matrix"]
            B_i = factor_dict["B_matrix"]
            delta_i = (B_i @ A_i) * lora_scaling_factor_si
            module_to_delta_list.setdefault(module_name, []).append(delta_i)

        classifier_weight_i, classifier_bias_i = extract_classifier_tensors(adapter_state)
        classifier_row_sources.append((class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i))

    with torch.no_grad():
        for module_name, delta_list_for_this_module in module_to_delta_list.items():
            merged_delta = ties_disjoint_merge(
                delta_list_for_this_module,
                density=density,
                sign_mode=sign_mode
            )
            target_weight_module = get_submodule_by_name(base_model_W0, module_name)

            if target_weight_module.weight.shape != merged_delta.shape:
                raise ValueError(
                    f"Shape mismatch for {module_name}: "
                    f"weight={target_weight_module.weight.shape}, delta={merged_delta.shape}"
                )

            target_weight_module.weight.data += merged_delta.to(
                target_weight_module.weight.device,
                dtype=target_weight_module.weight.dtype
            )

        # stitched classifier stays exactly as in your AVG merge line
        if hasattr(base_model_W0, "classifier"):
            for class_ids_for_this_adapter, clf_w_i, clf_b_i in classifier_row_sources:
                if clf_w_i is None:
                    continue

                for row_idx in class_ids_for_this_adapter:
                    base_model_W0.classifier.weight.data[row_idx] = clf_w_i[row_idx].to(
                        base_model_W0.classifier.weight.device,
                        dtype=base_model_W0.classifier.weight.dtype
                    )
                    if clf_b_i is not None and base_model_W0.classifier.bias is not None:
                        base_model_W0.classifier.bias.data[row_idx] = clf_b_i[row_idx].to(
                            base_model_W0.classifier.bias.device,
                            dtype=base_model_W0.classifier.bias.dtype
                        )

    return base_model_W0


# ============================================================
# Build models
# ============================================================
fixed_avg_model = build_fixed_avg_merged_model(
    base_model_dir=STEP1_FINAL_OUT,
    adapter_dirs=ADAPTER_DIRS,
    adapter_class_groups=ADAPTER_CLASS_GROUPS,
)

ties_model = build_ties_merged_model(
    base_model_dir=STEP1_FINAL_OUT,
    adapter_dirs=ADAPTER_DIRS,
    adapter_class_groups=ADAPTER_CLASS_GROUPS,
    density=TIES_DENSITY,
    sign_mode=TIES_SIGN_MODE,
)

print("Built Fixed AVG model.")
print(f"Built TIES model with density={TIES_DENSITY}, sign_mode={TIES_SIGN_MODE}")

In [ ]:
print("\n===== EVALUATE TIES MERGE =====")

ties_final_first = trainer_joint.evaluate(test_step1)
ties_final_later = trainer_joint.evaluate(test_later)
ties_final_seen  = trainer_joint.evaluate(test_seen)

print("\n[TIES] FINAL")
print("first_step :", ties_final_first)
print("later_steps:", ties_final_later)
print("all_seen   :", ties_final_seen)

In [ ]:
import os
import torch
from peft import PeftConfig

print("\n===== ITERATIVE MERGE: delta-space + stitched classifier =====")

def load_adapter_state_iter(adapter_dir):
    safe_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    bin_path = os.path.join(adapter_dir, "adapter_model.bin")
    if os.path.exists(safe_path):
        from safetensors.torch import load_file
        return load_file(safe_path)
    elif os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter weights found in {adapter_dir}")

def get_lora_scaling_factor_iter(adapter_dir):
    peft_cfg = PeftConfig.from_pretrained(adapter_dir)
    return peft_cfg.lora_alpha / peft_cfg.r

def extract_lora_factor_matrices_iter(adapter_state):
    lora_factor_matrices = {}
    for param_name, param_tensor in adapter_state.items():
        if ".lora_A." in param_name:
            base_module_name = param_name.split(".lora_A.")[0]
            lora_factor_matrices.setdefault(base_module_name, {})["A_matrix"] = param_tensor.detach().cpu().float()
        elif ".lora_B." in param_name:
            base_module_name = param_name.split(".lora_B.")[0]
            lora_factor_matrices.setdefault(base_module_name, {})["B_matrix"] = param_tensor.detach().cpu().float()

    lora_factor_matrices = {
        module_name: factors
        for module_name, factors in lora_factor_matrices.items()
        if "A_matrix" in factors and "B_matrix" in factors
    }
    return lora_factor_matrices

def extract_classifier_tensors_iter(adapter_state):
    classifier_weight_key = None
    classifier_bias_key = None
    for param_name in adapter_state.keys():
        if "classifier" in param_name and param_name.endswith("weight"):
            classifier_weight_key = param_name
        if "classifier" in param_name and param_name.endswith("bias"):
            classifier_bias_key = param_name

    if classifier_weight_key is None:
        return None, None

    classifier_weight_tensor = adapter_state[classifier_weight_key].detach().cpu().float()
    classifier_bias_tensor = (
        adapter_state[classifier_bias_key].detach().cpu().float()
        if classifier_bias_key is not None else None
    )
    return classifier_weight_tensor, classifier_bias_tensor

def merge_two_delta_dicts(delta_dict_a, delta_dict_b, wa=0.5, wb=0.5):
    merged = {}
    keys = sorted(set(delta_dict_a.keys()).union(set(delta_dict_b.keys())))
    for k in keys:
        if k not in delta_dict_a:
            merged[k] = wb * delta_dict_b[k]
        elif k not in delta_dict_b:
            merged[k] = wa * delta_dict_a[k]
        else:
            merged[k] = wa * delta_dict_a[k] + wb * delta_dict_b[k]
    return merged

def build_iterative_model_from_delta_dict(base_model_dir, delta_dict, classifier_sources):
    model = AutoModelForImageClassification.from_pretrained(base_model_dir)
    model.eval()

    with torch.no_grad():
        for module_name, delta in delta_dict.items():
            target_module = get_submodule_by_name(model, module_name)
            if target_module.weight.shape != delta.shape:
                raise ValueError(
                    f"Shape mismatch for {module_name}: "
                    f"weight={target_module.weight.shape}, delta={delta.shape}"
                )
            target_module.weight.data += delta.to(
                target_module.weight.device,
                dtype=target_module.weight.dtype
            )

    with torch.no_grad():
        for class_ids_for_this_adapter, classifier_weight_i, classifier_bias_i in classifier_sources:
            if classifier_weight_i is None:
                continue
            model.classifier.weight.data[class_ids_for_this_adapter] = classifier_weight_i[class_ids_for_this_adapter].to(
                model.classifier.weight.device,
                dtype=model.classifier.weight.dtype
            )
            if classifier_bias_i is not None and model.classifier.bias is not None:
                model.classifier.bias.data[class_ids_for_this_adapter] = classifier_bias_i[class_ids_for_this_adapter].to(
                    model.classifier.bias.device,
                    dtype=model.classifier.bias.dtype
                )

    model.eval()
    return model

def extract_adapter_delta_dict_iter(adapter_dir):
    adapter_state = load_adapter_state_iter(adapter_dir)
    scaling = get_lora_scaling_factor_iter(adapter_dir)
    lora_pairs = extract_lora_factor_matrices_iter(adapter_state)

    delta_dict = {}
    for module_name, factor_dict in lora_pairs.items():
        A_i = factor_dict["A_matrix"]
        B_i = factor_dict["B_matrix"]
        delta_dict[module_name] = (B_i @ A_i) * scaling

    cls_w, cls_b = extract_classifier_tensors_iter(adapter_state)
    return delta_dict, cls_w, cls_b

delta_1, clsw_1, clsb_1 = extract_adapter_delta_dict_iter(L1_dir)
delta_2, clsw_2, clsb_2 = extract_adapter_delta_dict_iter(L2_dir)
delta_3, clsw_3, clsb_3 = extract_adapter_delta_dict_iter(L3_dir)
delta_4, clsw_4, clsb_4 = extract_adapter_delta_dict_iter(L4_dir)

delta_12 = merge_two_delta_dicts(delta_1, delta_2, wa=0.5, wb=0.5)
delta_123 = merge_two_delta_dicts(delta_12, delta_3, wa=0.5, wb=0.5)
delta_1234 = merge_two_delta_dicts(delta_123, delta_4, wa=0.5, wb=0.5)

model_L1234_iterative = build_iterative_model_from_delta_dict(
    base_model_dir=B0_dir,
    delta_dict=delta_1234,
    classifier_sources=[
        (classes_for_step(1), clsw_1, clsb_1),
        (classes_for_step(2), clsw_2, clsb_2),
        (classes_for_step(3), clsw_3, clsb_3),
        (classes_for_step(4), clsw_4, clsb_4),
    ],
)

print("Built ITERATIVE model_L1234_iterative")

args_lora_eval_iter = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_lora_eval_iterative",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_lora_iter_final = Trainer(
    model=model_L1234_iterative,
    args=args_lora_eval_iter,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

iter_test_first = make_eval_dataset(first_step_classes, name="iterative_first")
iter_test_later = make_eval_dataset(later_step_classes, name="iterative_later")
iter_test_seen  = make_eval_dataset(final_seen_classes, name="iterative_seen")

iterative_final_first = trainer_lora_iter_final.evaluate(iter_test_first)
iterative_final_later = trainer_lora_iter_final.evaluate(iter_test_later)
iterative_final_seen  = trainer_lora_iter_final.evaluate(iter_test_seen)

print("Iterative Merged LoRA final - first_step:", iterative_final_first)
print("Iterative Merged LoRA final - later_steps:", iterative_final_later)
print("Iterative Merged LoRA final - all_seen:", iterative_final_seen)

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []

for s in range(2, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_ft"
    stale_final_dir = f"outputs/{RUN_NAME}/step_{s}_ft_final"
    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_final_dir):
        shutil.rmtree(stale_final_dir)

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print(f"\n[FT] Step {step_idx+1}")
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    tr_min, tr_max = _label_range(train_step)
    te_min, te_max = _label_range(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)
    print("Num labels:", num_classes)

    print("Loaded base_model_dir:", base_model_dir)
    ft_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print("Loaded FT model num_labels:", ft_model.config.num_labels)
    print("Loaded FT classifier out_features:", ft_model.classifier.out_features)

    assert ft_model.config.num_labels == num_classes, (
        f"Loaded FT model num_labels={ft_model.config.num_labels}, expected {num_classes}"
    )
    assert ft_model.classifier.out_features == num_classes, (
        f"Loaded FT classifier out_features={ft_model.classifier.out_features}, expected {num_classes}"
    )

    assert tr_min >= 0
    assert tr_max < ft_model.classifier.out_features, (
        f"Train labels out of range: max label {tr_max}, classifier out_features {ft_model.classifier.out_features}"
    )
    assert te_min >= 0
    assert te_max < ft_model.classifier.out_features, (
        f"Test labels out of range: max label {te_max}, classifier out_features {ft_model.classifier.out_features}"
    )

    step_ft_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft"

    args_ft = TrainingArguments(
        output_dir=step_ft_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft.train()
    eval_current = trainer_ft.evaluate(test_step)

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    os.makedirs(step_ft_dir, exist_ok=True)

    print(f"[FT] Step {step_idx+1} saving model to {step_ft_dir}")
    trainer_ft.save_model(step_ft_dir)
    image_processor.save_pretrained(step_ft_dir)
    print(f"[FT] Step {step_idx+1} save finished")

    reloaded_ft = AutoModelForImageClassification.from_pretrained(step_ft_dir)
    print(f"[FT] Reload check step {step_idx+1} num_labels:", reloaded_ft.config.num_labels)
    print(f"[FT] Reload check step {step_idx+1} classifier out_features:", reloaded_ft.classifier.out_features)

    assert reloaded_ft.config.num_labels == num_classes, (
        f"Reloaded saved FT num_labels={reloaded_ft.config.num_labels}, expected {num_classes}"
    )
    assert reloaded_ft.classifier.out_features == num_classes, (
        f"Reloaded saved FT classifier out_features={reloaded_ft.classifier.out_features}, expected {num_classes}"
    )

    base_model_dir = step_ft_dir

    seen_classes_now = classes_for_cumulative(step_idx)
    eval_first = make_eval_dataset(first_step_classes, name=f"ft_step{step_idx+1}_first_step")

    later_seen_now = [c for c in seen_classes_now if c not in first_step_classes]
    eval_later = make_eval_dataset(later_seen_now, name=f"ft_step{step_idx+1}_later_seen") if len(later_seen_now) > 0 else None
    eval_seen = make_eval_dataset(seen_classes_now, name=f"ft_step{step_idx+1}_all_seen")

    metrics_first = trainer_ft.evaluate(eval_first)
    metrics_seen = trainer_ft.evaluate(eval_seen)

    if eval_later is not None:
        metrics_later = trainer_ft.evaluate(eval_later)
    else:
        metrics_later = {"eval_accuracy": np.nan, "eval_loss": np.nan}

    ft_results.extend([
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(metrics_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_first.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(metrics_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_later.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(metrics_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_seen.get("eval_loss", np.nan)),
        },
    ])

print("\nFull FT continual training finished.")

In [ ]:
final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(final_seen_classes)

print("Final FT num_labels:", final_ft_model.config.num_labels)
print("Final FT classifier out_features:", final_ft_model.classifier.out_features)
assert final_ft_model.classifier.out_features == num_classes

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_config(config_joint)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

first_step_classes = classes_for_step(0)

later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))

final_seen_classes = classes_for_cumulative(num_steps - 1)

joint_eval_first = make_eval_dataset(first_step_classes, name="joint_first")
joint_eval_later = make_eval_dataset(later_step_classes, name="joint_later")
joint_eval_seen  = make_eval_dataset(final_seen_classes, name="joint_seen")

joint_final_first = trainer_joint.evaluate(eval_dataset=joint_eval_first)
joint_final_later = trainer_joint.evaluate(eval_dataset=joint_eval_later)
joint_final_seen  = trainer_joint.evaluate(eval_dataset=joint_eval_seen)

print("Joint final - first_step:", joint_final_first)
print("Joint final - later_steps:", joint_final_later)
print("Joint final - all_seen:", joint_final_seen)

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_all = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all classes:", joint_final_all)

## 11) Compare results (step test vs full test)

In [ ]:
# =========================
# FINAL RESULTS COLLECTION
# =========================

all_results = []

# Fixed AVG
all_results.extend([
    {
        "experiment": "fixed_avg_final_eval",
        "method": "lora_fixed_avg",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(merged_fixed_first),
        "eval_loss": grab_loss(merged_fixed_first),
    },
    {
        "experiment": "fixed_avg_final_eval",
        "method": "lora_fixed_avg",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(merged_fixed_later),
        "eval_loss": grab_loss(merged_fixed_later),
    },
    {
        "experiment": "fixed_avg_final_eval",
        "method": "lora_fixed_avg",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(merged_fixed_seen),
        "eval_loss": grab_loss(merged_fixed_seen),
    },
])

# TIES
all_results.extend([
    {
        "experiment": "ties_final_eval",
        "method": "lora_ties",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(ties_final_first),
        "eval_loss": grab_loss(ties_final_first),
    },
    {
        "experiment": "ties_final_eval",
        "method": "lora_ties",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(ties_final_later),
        "eval_loss": grab_loss(ties_final_later),
    },
    {
        "experiment": "ties_final_eval",
        "method": "lora_ties",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(ties_final_seen),
        "eval_loss": grab_loss(ties_final_seen),
    },
])

# Full FT
all_results.extend([
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(ft_final_first),
        "eval_loss": grab_loss(ft_final_first),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(ft_final_later),
        "eval_loss": grab_loss(ft_final_later),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(ft_final_seen),
        "eval_loss": grab_loss(ft_final_seen),
    },
])

# Joint
all_results.extend([
    {
        "experiment": "joint_final_eval",
        "method": "joint_training",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(joint_final_first),
        "eval_loss": grab_loss(joint_final_first),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_training",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(joint_final_later),
        "eval_loss": grab_loss(joint_final_later),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_training",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(joint_final_seen),
        "eval_loss": grab_loss(joint_final_seen),
    },
])

results_df = pd.DataFrame(all_results)
display(results_df.head(20))
print("results_df shape:", results_df.shape)

In [ ]:
# =========================
# METHOD NAME MAPPING
# =========================

def map_method(x):
    if x == "lora_fixed_avg":
        return "Fixed AVG"
    elif x == "lora_ties":
        return "TIES"
    elif x == "full_finetune":
        return "Full FT"
    elif x == "joint_training":
        return "Joint"
    else:
        return x

def map_adapter_name(row):
    if row["method"] == "Fixed AVG":
        return "L1234_avg"
    elif row["method"] == "TIES":
        return "L1234_ties"
    elif row["method"] == "Full FT":
        return "Full FT"
    elif row["method"] == "Joint":
        return "Joint"
    return row["method"]

results_df["method"] = results_df["method"].apply(map_method)
results_df["adapter_name"] = results_df.apply(map_adapter_name, axis=1)

display(results_df.head(20))

In [ ]:
summary_lines = [
    "Experiment summary",
    "==================",
    "",
]

for _, row in results_df_clean.iterrows():
    acc = row["accuracy"]
    loss = row["loss"]

    acc_str = f"{acc:.4f}" if pd.notna(acc) else "nan"
    loss_str = f"{loss:.4f}" if pd.notna(loss) else "nan"

    summary_lines.append(
        f"method={row['method']} | adapter_name={row['adapter_name']} | "
        f"step={row['step']} | eval_stage={row['eval_stage']} | "
        f"eval_set={row['eval_set']} | accuracy={acc_str} | loss={loss_str}"
    )

with open(os.path.join(RESULTS_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))

In [ ]:
# =========================
# FINAL COMPARISON PLOT
# =========================

df_final = results_df.copy()

df_final = df_final[df_final["method"].isin([
    "Fixed AVG",
    "TIES",
    "Full FT",
    "Joint"
])]

method_order = ["Fixed AVG", "TIES", "Full FT", "Joint"]
eval_order = ["first_step", "later_steps", "all_seen"]

df_final["method"] = pd.Categorical(df_final["method"], categories=method_order, ordered=True)
df_final["eval_type"] = pd.Categorical(df_final["eval_type"], categories=eval_order, ordered=True)

df_final = df_final.sort_values(["eval_type", "method"])

plt.figure(figsize=(10, 6))

x = np.arange(len(eval_order))
width = 0.18

for i, method_name in enumerate(method_order):
    vals = []
    for eval_name in eval_order:
        sub = df_final[(df_final["method"] == method_name) & (df_final["eval_type"] == eval_name)]
        vals.append(sub["eval_accuracy"].values[0] if len(sub) > 0 else np.nan)

    plt.bar(x + (i - (len(method_order)-1)/2) * width, vals, width=width, label=method_name)

plt.xticks(x, ["First Step", "Later Steps", "All Seen"])
plt.ylabel("Accuracy")
plt.title("Final Accuracy Comparison")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "final_accuracy_comparison_avg_ties_ft_joint.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen  = df_final[df_final["eval_set"] == "all_seen"]

def make_barplot(df_sub, title, filename):
    methods = ["Merged LoRA", "Full FT", "Joint"]
    vals = []
    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(7, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

make_barplot(final_first, "Final Accuracy on B0 Classes", "final_first_step_accuracy_lora_ft_joint.png")
make_barplot(final_later, "Final Accuracy on Later Classes", "final_later_steps_accuracy_lora_ft_joint.png")
make_barplot(final_seen,  "Final Accuracy on All Seen Classes", "final_all_seen_accuracy_lora_ft_joint.png")

In [ ]:
plot_df = results_df_clean.copy()
plot_df_step = plot_df[plot_df["eval_stage"] == "step"].copy()

lora_current = plot_df_step[
    (plot_df_step["method"] == "LoRA separate") &
    (plot_df_step["eval_set"] == "current_step")
].copy()

plt.figure(figsize=(7, 4))
plt.plot(lora_current["step"], lora_current["accuracy"], marker="o")
plt.xticks(lora_current["step"], lora_current["adapter_name"])
plt.xlabel("Adapter")
plt.ylabel("Accuracy")
plt.title("LoRA Separate: Current-Step Accuracy over L1..L4")
plt.tight_layout()

out = os.path.join(PLOTS_DIR, "curve_lora_separate_current_step.png")
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out)

In [ ]:
def make_barplot(df_sub, title, filename):
    methods = [
        "Fixed AVG",
        "Fixed Weighted",
        "Iterative",
        "Full FT",
        "Joint"
    ]
    vals = []
    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(10, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen  = df_final[df_final["eval_set"] == "all_seen"]

make_barplot(final_first, "Final Accuracy on B0 Classes", "final_first_step_all_methods.png")
make_barplot(final_later, "Final Accuracy on Later Classes", "final_later_steps_all_methods.png")
make_barplot(final_seen,  "Final Accuracy on All Seen Classes", "final_all_seen_all_methods.png")

In [ ]:
# =========================
# MERGE-ONLY COMPARISON
# =========================

df_merge_only = results_df[results_df["method"].isin([
    "Fixed AVG",
    "TIES"
])].copy()

method_order_merge = ["Fixed AVG", "TIES"]
eval_order = ["first_step", "later_steps", "all_seen"]

df_merge_only["method"] = pd.Categorical(df_merge_only["method"], categories=method_order_merge, ordered=True)
df_merge_only["eval_type"] = pd.Categorical(df_merge_only["eval_type"], categories=eval_order, ordered=True)

df_merge_only = df_merge_only.sort_values(["eval_type", "method"])

plt.figure(figsize=(8, 6))

x = np.arange(len(eval_order))
width = 0.28

for i, method_name in enumerate(method_order_merge):
    vals = []
    for eval_name in eval_order:
        sub = df_merge_only[(df_merge_only["method"] == method_name) & (df_merge_only["eval_type"] == eval_name)]
        vals.append(sub["eval_accuracy"].values[0] if len(sub) > 0 else np.nan)

    plt.bar(x + (i - (len(method_order_merge)-1)/2) * width, vals, width=width, label=method_name)

plt.xticks(x, ["First Step", "Later Steps", "All Seen"])
plt.ylabel("Accuracy")
plt.title("Merge-Only Final Accuracy Comparison")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "merge_only_avg_vs_ties.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
# =========================
# MERGE SUMMARY TABLE
# =========================

merge_summary = results_df[results_df["method"].isin([
    "Fixed AVG",
    "TIES"
])].copy()

merge_summary = merge_summary.pivot_table(
    index="method",
    columns="eval_type",
    values="eval_accuracy",
    aggfunc="first"
).reset_index()

merge_summary = merge_summary.rename(columns={
    "first_step": "acc_first_step",
    "later_steps": "acc_later_steps",
    "all_seen": "acc_all_seen",
})

method_order_merge = ["Fixed AVG", "TIES"]
merge_summary["method"] = pd.Categorical(merge_summary["method"], categories=method_order_merge, ordered=True)
merge_summary = merge_summary.sort_values("method").reset_index(drop=True)

display(merge_summary)

summary_path = os.path.join(TABLES_DIR, "merge_summary_avg_vs_ties.csv")
merge_summary.to_csv(summary_path, index=False)
print("Saved summary to:", summary_path)